#自动通过PubChem API抓取化合物SMILE编码并进行3D建模

In [3]:
import requests
import time
from rdkit import Chem
from rdkit.Chem import AllChem
import py3Dmol
#通过API抓取化合物SMILE
def show_me_3d(compound_name):
    print(f'正在获取{compound_name}的SMILE...')
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES/JSON"
        response = requests.get(url,timeout=5)
        data = response.json()
        smiles = data['PropertyTable']['Properties'][0]['ConnectivitySMILES']
        print(f'该化合物的SMILE为：{smiles}')

    except Exception as e:
        return "Not Found" #因为还要继续所以不return smiles了
    #将抓取到的SMILE转化成sdf分子信息
    mol = Chem.MolFromSmiles(smiles)
    m2 = Chem.AddHs(mol)
    print("正在进行3D建模...")
    if AllChem.EmbedMolecule(m2) != 0:
        print("建模失败！")
        return

    AllChem.MMFFOptimizeMolecule(m2) #力场优化
    #接通py3Dmol
    mol_block = Chem.MolToMolBlock(m2)
    p_viewer = py3Dmol.view(width=400,height=400)
    p_viewer.addModel(mol_block,'mol')
    p_viewer.setStyle({'stick': {'radius':0.2}})
    p_viewer.zoomTo()
    p_viewer.spin()

    print("建模完毕！")
    return p_viewer.show()
                     
    


In [5]:
show_me_3d("caffeine")

正在获取caffeine的SMILE...
该化合物的SMILE为：CN1C=NC2=C1C(=O)N(C(=O)N2C)C
正在进行3D建模...
建模完毕！


3Dmol.js failed to load for some reason. Please check your browser console for error messages.